## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

## **Imports**

In [3]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import scipy.sparse as sps
from sklearn.cluster import KMeans
import gc

from Challenge.paths import load_xgboost_cv_folds, XGBOOST_MODELS, XGBOOST_DATAFRAMES
from Challenge.utils import load_models

Running on local — storage at: /home/luigi/RecSys


## **Load Data**

In [4]:
URM_inner, URM_outer, folds = load_xgboost_cv_folds()

## **Recommeder List**

In [5]:
from Recommenders.NonPersonalizedRecommender import TopPop
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_WARP_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_BPR_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_SVDpp_Cython

from Recommenders.SLIM.Cython.SLIM_BPR_Cython import SLIM_BPR_Cython
from Recommenders.MatrixFactorization.NMFRecommender import NMFRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask
from implicit.cpu.als import AlternatingLeastSquares

models_mapping = {
    'TopPop': TopPop,
    'ItemKNN_cosine': ItemKNNCFRecommender,
    'ItemKNN_jaccard': ItemKNNCFRecommender,
    'ItemKNN_asymmetric': ItemKNNCFRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'ItemKNN_dice': ItemKNNCFRecommender,
    'UserKNN_cosine': UserKNNCFRecommender,
    'UserKNN_jaccard': UserKNNCFRecommender,
    'UserKNN_asymmetric': UserKNNCFRecommender,
    'UserKNN_tversky': UserKNNCFRecommender,
    'UserKNN_dice': UserKNNCFRecommender,
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'EASE_R': EASE_R_Recommender,
    'P3alpha': P3alphaRecommender,
    'RP3beta': RP3betaRecommender,
    'IALS': AlternatingLeastSquares,
    'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython,
    'MatrixFactorization_BPR': MatrixFactorization_BPR_Cython,
    # 'MatrixFactorization_SVDpp': MatrixFactorization_SVDpp_Cython,
    
    # 'SLIM_BPR': SLIM_BPR_Cython,
    'NMF': NMFRecommender,
    'MultVAE': MultVAERecommender_PyTorch_OptimizerMask,
}

candidate_mapping = {
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'MultVAE': MultVAERecommender_PyTorch_OptimizerMask,
    'IALS': AlternatingLeastSquares,
    'RP3beta': RP3betaRecommender,
    'UserKNN_tversky': UserKNNCFRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython,
    'TopPop': TopPop,
}

candidate_cutoff = {
    'SLIMElasticNet': 80,
    'MultVAE': 60,
    'IALS': 60,
    'RP3beta': 20,
    'UserKNN_tversky': 30,
    'ItemKNN_tversky': 30,
    'MatrixFactorization_WARP': 30,
    'TopPop': 50,
}

## **Functions**

In [6]:
def get_user_batches(user_ids, batch_size=1000):
    for i in range(0, len(user_ids), batch_size):
        yield user_ids[i:i + batch_size]

In [7]:
def generate_candidates(URM_train, user_ids, models_mapping, models_cutoff, folder):
    estimated_rows = len(user_ids) * sum(models_cutoff.values())
    
    all_users = np.empty(estimated_rows, dtype=np.int16) # 27k users fit in int16 (32k limit)
    all_items = np.empty(estimated_rows, dtype=np.int16) #  7k items fit in int16 (32k limit)
    
    current_ptr = 0

    for model_name, model in load_models(URM_train, models_mapping, folder, verbose=False):
        print(f"Computing candidates with {model_name}...")

        cutoff = models_cutoff.get(model_name, 0)
        assert cutoff > 0, f"Cutoff not defined for model {model_name}" 

        if type(model) == AlternatingLeastSquares:
            recommended_items, scores = model.recommend(
                user_ids, 
                URM_train, 
                N=cutoff, 
                filter_already_liked_items=True
            )

        elif type(model) == MultVAERecommender_PyTorch_OptimizerMask:
            # Do in batches to avoid OOM in GPU
            recommended_items = []
            for user_batch in get_user_batches(user_ids, batch_size=500):
                batch_recs = model.recommend(user_batch, cutoff=cutoff)
                recommended_items.extend(batch_recs)
            recommended_items = recommended_items

        else:
            recommended_items = model.recommend(user_ids, cutoff=cutoff)

        # --- VECTORIZED FLATTENING ---
        # recs is likely shape (n_users, cutoff)
        # We need to transform this into:
        # Users: [1, 1, 1, 2, 2, 2...]
        # Items: [A, B, C, X, Y, Z...]
        
        recs = np.array(recommended_items) # Ensure numpy array
        
        # Flatten items
        flat_items = recs.flatten()
        
        # Create corresponding users array
        # Repeat every user_id N times, where N is the number of columns (cutoff)
        # shape[1] covers cases where model might return fewer than cutoff
        flat_users = np.repeat(user_ids, recs.shape[1])
        
        # Fill the pre-allocated arrays
        num_new = len(flat_items)
        end_ptr = current_ptr + num_new
        
        all_users[current_ptr:end_ptr] = flat_users
        all_items[current_ptr:end_ptr] = flat_items
        
        current_ptr = end_ptr

    # TRIM & DEDUPLICATE
    # Cut arrays to actual filled size
    all_users = all_users[:current_ptr]
    all_items = all_items[:current_ptr]
    
    # Stack into (N, 2) matrix -> [[u1, i1], [u1, i2]...]
    candidates_matrix = np.vstack((all_users, all_items)).T
    
    # np.unique with axis=0 finds unique ROWS. This is extremely fast.
    unique_candidates = np.unique(candidates_matrix, axis=0)
    
    # CREATE DATAFRAME
    # We only touch Pandas at the very end
    candidates_df = pd.DataFrame(unique_candidates, columns=['UserID', 'ItemID'])
    
    # Sort for faster joining later
    candidates_df.sort_values(by=['UserID', 'ItemID'], inplace=True)
    
    return candidates_df

In [8]:
def add_models_features(df: pd.DataFrame, URM: sps.csr_matrix, models_mapping, model_folder):
    assert 'UserID' in df.columns and 'ItemID' in df.columns, "DataFrame must contain 'UserID' and 'ItemID' columns"
    assert df['UserID'].is_monotonic_increasing, "DataFrame 'UserID' column must be sorted in ascending order"

    model_folder = os.path.join(XGBOOST_MODELS, model_folder)
    models = load_models(URM, models_mapping, model_folder=model_folder)

    cand_users = df['UserID'].values
    cand_items = df['ItemID'].values
    unique_users = df['UserID'].unique()

    # Map Global UserIDs to Local Matrix Indices (0 to N_unique)
    # Since df is sorted, unique_users is sorted. searchsorted is fast.
    # This array tells us: "For row i in df, which row in the scores matrix should I look at?"
    local_user_indices = np.searchsorted(unique_users, cand_users)

    # unique users count should be all the users in df
    # but for safety we subset URM
    URM_subset = URM[unique_users]

    # Pre-calculate seen indices relative to the subset
    # seen_rows will be 0..N_unique-1, matching the scores matrix
    seen_rows, seen_cols = URM_subset.nonzero()

    for label, recommender in models:
        print(f"Processing features for model: {label}")

        if type(recommender) == MultVAERecommender_PyTorch_OptimizerMask:
            # Do in batches to avoid OOM in GPU
            scores = np.zeros((len(unique_users), n_items), dtype=np.float32)
            start_idx = 0
            for user_batch in get_user_batches(unique_users, batch_size=500):
                batch_scores = recommender._compute_item_score(user_id_array=user_batch)
                end_idx = start_idx + len(user_batch)
                scores[start_idx:end_idx] = batch_scores
                start_idx = end_idx
        
        elif type(recommender) == AlternatingLeastSquares:
            recommended_items, scores = recommender.recommend(
                unique_users,
                URM_subset,
                N=URM_subset.shape[1],
                filter_already_liked_items=False
            )

            assert len(recommended_items) == len(scores) == len(unique_users), "Mismatch in recommended items and scores length"
            assert scores.shape[1] == URM_subset.shape[1], "Scores shape does not match number of items in URM"

            # Simple Fix: Un-sort the results
            # rec_items contains ItemIDs. argsort gives us the indices that would sort those IDs (0, 1, 2...)
            sorted_indices = np.argsort(recommended_items, axis=1)
            
            # We use take_along_axis to apply those indices to the scores
            # This aligns the scores so Column 0 is ItemID 0, Column 1 is ItemID 1...
            scores = np.take_along_axis(scores, sorted_indices, axis=1)
        
        else:
            scores = recommender._compute_item_score(user_id_array=unique_users)

        scores = np.array(scores)  # Ensure numpy array

        # Normalize
        norm_factor = np.linalg.norm(scores, np.inf, axis=1, keepdims=True)
        norm_factor[norm_factor == 0] = 1.0 
        linf_scores = scores / norm_factor

        # Remove seen items        
        # Set those specific entries to -inf
        linf_scores[seen_rows, seen_cols] = -np.inf

        # Calculate Ranks
        # argsort sorts ascending, so we use [::-1] to get descending (highest score first)
        # This returns INDICES of items. 
        # shape: (user_count, n_items)
        rank_order = np.argsort(linf_scores, axis=1)[:, ::-1]
        
        # We need the inverse mapping: item_id -> rank_position
        n_scores, n_items = linf_scores.shape
        rank_matrix = np.empty((n_scores, n_items), dtype=np.int32)
        
        # Fancy numpy trick to invert the permutation vectors in one go
        # Arrays of shape (n_scores, 1) needed for broadcasting
        row_indices = np.arange(n_scores)[:, None] 
        rank_matrix[row_indices, rank_order] = np.arange(n_items)
        
        # Extract Values for Candidate Pairs
        scores_final = linf_scores[local_user_indices, cand_items]
        ranks_final = rank_matrix[local_user_indices, cand_items]
        
        # Assign directly to DF
        df[f"{label}_Score"] = scores_final
        df[f"{label}_RankPosition"] = ranks_final
        df[f"{label}_Recommended"] = (ranks_final < 20).astype(int)
    
        # Free memory immediately
        del scores, linf_scores, rank_matrix, rank_order, scores_final, ranks_final
        gc.collect()

    return df
        

In [9]:
def calculate_item_item_features_fast(df, URM, model_folder):
    # Check df is sorted by UserID
    assert df['UserID'].is_monotonic_increasing, "DataFrame must be sorted by UserID in increasing order."

    # Load only distinct model types
    similarity_models = load_models(
        URM,
        {
            'ItemKNN_tversky': ItemKNNCFRecommender, 
            'RP3beta': RP3betaRecommender,
            'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender 
        },
        model_folder=os.path.join(XGBOOST_MODELS, model_folder)
    )

    # Pre-calculate mapping: UserID -> [List of Candidate ItemIDs]
    # We use numpy split for max speed
    user_ids = df['UserID'].values
    item_ids = df['ItemID'].values
    
    # Find indices where user changes
    unique_users, user_starts = np.unique(user_ids, return_index=True)
    # Map UserID to (start_index, end_index) in the sorted arrays
    user_map = {}
    for i, user_id in enumerate(unique_users):
        start = user_starts[i]
        end = user_starts[i+1] if i + 1 < len(unique_users) else len(user_ids)
        user_map[user_id] = (start, end)
    
    # Prepare result dictionary
    N_ROWS = len(df)
    
    for label, recommender in similarity_models:
        print(f"Extracting MAX similarity for {label}...")
        
        # Get Sparse Matrix
        W_sparse = recommender.W_sparse
        if not sps.issparse(W_sparse):
            W_sparse = sps.csr_matrix(W_sparse)
            
        # Feature arrays
        max_sims = np.zeros(N_ROWS, dtype=np.float32)
        std_sims = np.zeros(N_ROWS, dtype=np.float32)
        
        # Iterate over unique users in the candidates
        for user_id in tqdm(unique_users):
            start, end = user_map[user_id]
            
            # Get Seen Items for this user (Indices)
            seen_items = URM.indices[URM.indptr[user_id]:URM.indptr[user_id+1]]
            
            if len(seen_items) == 0:
                continue
                
            # Get Candidate Items for this user
            cand_items = item_ids[start:end]
            
            # --- THE CORE OPTIMIZATION ---
            # Instead of .toarray(), we slice sparse matrix
            # Submatrix: Rows=Candidates, Cols=SeenItems
            # This is efficient because W is CSR (fast row slicing)
            sub_W = W_sparse[cand_items, :][:, seen_items]
            
            # Check if sub_W is effectively empty
            if sub_W.nnz > 0:
                # Max similarity to any seen item
                max_sims[start:end] = sub_W.max(axis=1).toarray().flatten()
                
                # For Std, you usually need to densify (slower)
                dense_batch = sub_W.toarray()
                std_sims[start:end] = dense_batch.std(axis=1)

        # Assign directly to DF
        df[f'{label}_MaxSim'] = max_sims
        df[f'{label}_StdSim'] = std_sims
    
    return df

In [10]:
def add_embedding_features(df, model_folder, 
                            n_item_clusters=20, n_user_clusters=30, 
                            batch_size=500, seed=42):
    # Load ALS model
    model_path = os.path.join(XGBOOST_MODELS, model_folder, "IALS.npz")
    recommender = AlternatingLeastSquares.load(model_path)

    # Get factors from trained ALS model
    user_factors = recommender.user_factors
    item_factors = recommender.item_factors
    
    # Cluster Users
    kmeans_users = KMeans(n_clusters=n_user_clusters, random_state=seed)
    user_clusters = kmeans_users.fit_predict(user_factors)
    user_centroid_matrix = kmeans_users.cluster_centers_.astype(np.float32)

    # Cluster Items
    kmeans_items = KMeans(n_clusters=n_item_clusters, random_state=seed)
    item_clusters = kmeans_items.fit_predict(item_factors)
    item_centroid_matrix = kmeans_items.cluster_centers_.astype(np.float32)

    # --- 3. PRE-ALLOCATE RESULT ARRAYS ---
    # We allocate arrays for the full length of DF to avoid DataFrame fragmentation
    N_ROWS = len(df)
    
    # Output arrays
    res_u_cluster = np.zeros(N_ROWS, dtype=np.int16)
    res_i_cluster = np.zeros(N_ROWS, dtype=np.int16)
    res_inter_cluster = np.zeros(N_ROWS, dtype=np.int16)
    res_u_dist = np.zeros(N_ROWS, dtype=np.float32)
    res_i_dist = np.zeros(N_ROWS, dtype=np.float32)
    res_cross_dist = np.zeros(N_ROWS, dtype=np.float32)

    # Input arrays from DF (Read-only)
    all_user_ids = df['UserID'].values
    all_item_ids = df['ItemID'].values

    # --- 4. BATCHED FEATURE CALCULATION ---
    print(f"Computing distances in batches of {batch_size}...")
    
    for start in tqdm(range(0, N_ROWS, batch_size)):
        end = min(start + batch_size, N_ROWS)
        
        # A. Get IDs for this batch
        batch_u_ids = all_user_ids[start:end]
        batch_i_ids = all_item_ids[start:end]
        
        # B. Map to Clusters
        batch_u_clusters = user_clusters[batch_u_ids]
        batch_i_clusters = item_clusters[batch_i_ids]
        
        # Store Clusters
        res_u_cluster[start:end] = batch_u_clusters
        res_i_cluster[start:end] = batch_i_clusters
        res_inter_cluster[start:end] = batch_u_clusters * n_item_clusters + batch_i_clusters

        # C. Retrieve Factors & Centroids for this batch
        # Advanced Indexing -> Creates temporary small arrays (size of batch)
        # 1. User Vector vs User Centroid
        u_vecs = user_factors[batch_u_ids]
        u_cents = user_centroid_matrix[batch_u_clusters]
        
        # 2. Item Vector vs Item Centroid
        i_vecs = item_factors[batch_i_ids]
        i_cents = item_centroid_matrix[batch_i_clusters]
        
        # D. Calculate Distances
        # User eccentricity
        res_u_dist[start:end] = np.linalg.norm(u_vecs - u_cents, axis=1)
        
        # Item eccentricity
        res_i_dist[start:end] = np.linalg.norm(i_vecs - i_cents, axis=1)
        
        # Cross Distance (User Vector vs Item-Cluster Centroid)
        # This is the "Genre Affinity" score
        res_cross_dist[start:end] = np.linalg.norm(u_vecs - i_cents, axis=1)

    # --- 5. ASSIGN TO DATAFRAME ---
    print("Assigning columns to DataFrame...")
    df['User_Cluster'] = res_u_cluster
    df['Item_Cluster'] = res_i_cluster
    df['Cluster_Interaction'] = res_inter_cluster
    df['User_Cluster_Dist'] = res_u_dist
    df['Item_Cluster_Dist'] = res_i_dist
    df['User_to_ItemCluster_Dist'] = res_cross_dist
    
    return df

In [11]:
def add_aggregate_features_stats(df):
    # Consensus Features
    recommended_columns = [col for col in df.columns if col.endswith('_Recommended')]
    df['Counter_Recommended'] = df[recommended_columns].sum(axis=1).astype(int)

    # Rank Position Statistics
    position_columns = [col for col in df.columns if col.endswith('_RankPosition')]
    
    df['Mean_RankPosition'] = df[position_columns].mean(axis=1)
    df['Std_RankPosition'] = df[position_columns].std(axis=1)
    df['Skew_RankPosition'] = df[position_columns].skew(axis=1)
    df['Kurtosis_RankPosition'] = df[position_columns].kurtosis(axis=1)
    
    # Score Statistics
    score_columns = [col for col in df.columns if col.endswith('_Score')]

    df['Mean_Score'] = df[score_columns].mean(axis=1)
    df['Std_Score'] = df[score_columns].std(axis=1)
    df['Skew_Score'] = df[score_columns].skew(axis=1)
    df['Kurtosis_Score'] = df[score_columns].kurtosis(axis=1)
    
    return df

In [12]:
def add_user_stats(df, URM):
    # We need to map UserID and ItemID to the URM indices
    user_ids = df['UserID'].values
    item_ids = df['ItemID'].values
    
    # User Profile Length
    user_profile_len = np.ediff1d(URM.indptr)
    df['User_Profile_Len'] = user_profile_len[user_ids]
    
    # Item Global Popularity
    item_popularity = np.ediff1d(URM.tocsc().indptr)
    df['Item_Global_Popularity'] = item_popularity[item_ids]
    
    return df

In [ ]:
def sanity_check(df, verbose=True):
    print("--- STARTING SANITY CHECK ---")
    problems_found = False

    # 0. Check DataFrame is sorted by UserID
    if not df['UserID'].is_monotonic_increasing:
        print("\n[CRITICAL] DataFrame is not sorted by UserID in increasing order.")
        problems_found = True

    # 1. Check for Missing Values (NaN)
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print("\n[CRITICAL] NaN Values Found:")
        print(null_counts[null_counts > 0])
        problems_found = True
    else:
        if verbose: print("[OK] No NaNs found.")

    # 2. Check for Infinite Values (inf / -inf)
    # Common issue when normalizing by zero variance or dividing scores
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = np.isinf(df[numeric_cols]).sum()
    if inf_counts.sum() > 0:
        print("\n[CRITICAL] Infinite Values Found (Division by Zero?):")
        print(inf_counts[inf_counts > 0])
        problems_found = True
    else:
        if verbose: print("[OK] No Infinite values found.")

    # 3. Check for Duplicates (UserID, ItemID)
    # Stacking fails if you have multiple rows for the same User-Item pair
    if df.duplicated(subset=['UserID', 'ItemID']).any():
        n_dupes = df.duplicated(subset=['UserID', 'ItemID']).sum()
        print(f"\n[CRITICAL] Duplicate (UserID, ItemID) pairs found: {n_dupes}")
        problems_found = True
    else:
        if verbose: print("[OK] Keys (UserID, ItemID) are unique.")

    # 4. Check Data Types (IDs must be int)
    # Merges (pd.merge) often convert ints to float if there were missing keys initially
    if df['UserID'].dtype not in [int, np.int16, np.int32, np.int64]:
        print(f"\n[WARNING] UserID is {df['UserID'].dtype}, expected int. (Did a merge fail?)")
        # Auto-fix attempt
        # df['UserID'] = df['UserID'].astype(int) 
    
    if df['ItemID'].dtype not in [int, np.int16, np.int32, np.int64]:
        print(f"\n[WARNING] ItemID is {df['ItemID'].dtype}, expected int.")

    # 5. Check for Constant Columns (Zero Variance)
    # These crash some implementations of Normalization and add no info to XGBoost
    std_devs = df[numeric_cols].std()
    constant_cols = std_devs[std_devs == 0].index.tolist()
    if len(constant_cols) > 0:
        print("\n[WARNING] The following columns have ZERO variance (Constant values):")
        print(constant_cols)
        print("Recommendation: Drop them.")
    
    # 6. Check Logic (Ranks shouldn't be negative)
    rank_cols = [c for c in df.columns if 'Rank' in c and 'Skew' not in c and 'Kurtosis' not in c]
    if rank_cols:
        min_ranks = df[rank_cols].min()
        if (min_ranks < 0).any():
             print("\n[CRITICAL] Negative Ranks found (Logic Error):")
             print(min_ranks[min_ranks < 0])
             problems_found = True

    if problems_found:
        print("\n--- SANITY CHECK FAILED: Fix errors before training ---")
        # raise ValueError("Data Integrity Issues Found") # Uncomment to force stop
    else:
        print("\n--- SANITY CHECK PASSED: Data is clean ---")

    return not problems_found

In [14]:
def optimize_dataframe_types(df):
    # 1. Downcast Integers (UserID, ItemID, Ranks)
    # int64 -> int32 (saves 50% RAM)
    ints = df.select_dtypes(include=['int64', 'int32', 'int']).columns
    df[ints] = df[ints].apply(pd.to_numeric, downcast='integer')
    
    # 2. Downcast Floats (Scores, Similarities)
    # float64 -> float32 (saves 50% RAM, XGBoost doesn't need float64)
    floats = df.select_dtypes(include=['float64', 'float']).columns
    df[floats] = df[floats].apply(pd.to_numeric, downcast='float')
    
    return df

## **Training Dataframe**

In [15]:
for i, (URM_train, URM_val) in enumerate(folds):
    models_folder = os.path.join(XGBOOST_MODELS, "features", "folds", f"fold_{i}")

    # Generate Candidates
    print(f"\n=== FOLD {i}: Generating Candidates ===")
    df = generate_candidates(URM_train, 
                            user_ids=np.arange(URM_train.shape[0]), 
                            models_mapping=candidate_mapping, 
                            models_cutoff=candidate_cutoff,
                            folder=models_folder)
    print(f"Candidates Generated: {len(df)}")

    # Add Label Column
    print(f"\n=== FOLD {i}: Adding Labels ===")
    # Advanced Indexing: Query the matrix directly
    # "For every (User, Item) in df, give me the value in URM"
    # This returns a matrix, so we convert to array and flatten
    ground_truth_values = URM_val[df['UserID'].values, df['ItemID'].values]

    # Convert to Boolean/Int (1 if interaction exists, 0 otherwise)
    # We use np.array(..).squeeze() because sparse indexing returns a 2D structure
    df['Label'] = (np.array(ground_truth_values).squeeze() > 0).astype(np.int8)

    print(f"Positive Labels: {df['Label'].sum()} / {len(df)} ({100.0 * df['Label'].mean():.4f}%)")

    # Add Model Features
    print(f"\n=== FOLD {i}: Adding Model Features ===")
    df = add_models_features(df, URM_train, models_mapping, model_folder=models_folder)
    print(f"Features after model addition: {len(df.columns)}")

    # Add Item-Item Similarity Features
    print(f"\n=== FOLD {i}: Adding Item-Item Similarity Features ===")
    df = calculate_item_item_features_fast(df, URM_train, models_folder)
    print(f"Features after item-item similarity addition: {len(df.columns)}")

    # Add Embedding Features
    # Add Embedding Features
    print(f"\n=== FOLD {i}: Adding Embedding Features ===")
    df = add_embedding_features(df, model_folder=models_folder)
    print(f"Features after embedding addition: {len(df.columns)}")

    # Add Aggregate Features
    print(f"\n=== FOLD {i}: Adding Aggregate Features ===")
    df = add_aggregate_features_stats(df)
    print(f"Features after aggregate addition: {len(df.columns)}")

    # Add User Stats
    print(f"\n=== FOLD {i}: Adding User and Item Stats ===")
    df = add_user_stats(df, URM_train)
    print(f"Features after user/item stats addition: {len(df.columns)}")

    # Sanity Check
    print(f"\n=== FOLD {i}: Running Sanity Check ===")
    success = sanity_check(df)
    if not success:
        print(f"Sanity check failed for fold {i}.")

    # Optimize Data Types
    print(f"\n=== FOLD {i}: Optimizing Data Types ===")
    df = optimize_dataframe_types(df)
    
    # Save
    save_path = os.path.join(XGBOOST_DATAFRAMES, "training_data", f"fold_{i}.parquet")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    df.to_parquet(
        path=save_path,
        engine='fastparquet',
        compression='zstd',
        index=False
    )

    print(f"\n=== FOLD {i}: DataFrame saved to {save_path} ===")

    # Clean up
    del df
    gc.collect()


=== FOLD 0: Generating Candidates ===
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/folds/fold_0SLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Computing candidates with SLIMElasticNet...
Computing candidates with MultVAE...


/home/luigi/RecSys/Recommenders/Neural/MultVAE_PyTorch_Recommender.py:203: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  user_batch_tensor = torch.sparse_csr_tensor(
/home/luigi/.venvs/recsys/lib/python3.13/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Computing candidates with IALS...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/folds/fold_0RP3beta'
RP3betaRecommender: Loading complete
Computing candidates with RP3beta...
UserKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/folds/fold_0UserKNN_tversky'
UserKNNCFRecommender: Loading complete
Computing candidates with UserKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/folds/fold_0ItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Computing candidates with ItemKNN_tversky...
MatrixFactorization_WARP_Cython_Recommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/folds/fold_0MatrixFactorization_WARP'
MatrixFactorization_WARP_Cython_Recommender: Loading complete
Computing candidates with MatrixFactorization_WARP...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/fe

100%|██████████| 27095/27095 [00:04<00:00, 6664.41it/s]


Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/folds/fold_0RP3beta'
RP3betaRecommender: Loading complete
Extracting MAX similarity for RP3beta...


...

Unloading RP3beta...
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/folds/fold_4SLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Extracting MAX similarity for SLIMElasticNet...


100%|██████████| 27095/27095 [00:07<00:00, 3752.22it/s]


Unloading SLIMElasticNet...
Features after item-item similarity addition: 69

=== FOLD 4: Adding Embedding Features ===
Computing distances in batches of 500...


100%|██████████| 7906/7906 [00:01<00:00, 7749.23it/s]


Assigning columns to DataFrame...
Features after embedding addition: 75

=== FOLD 4: Adding Aggregate Features ===
Features after aggregate addition: 84

=== FOLD 4: Adding User and Item Stats ===
Features after user/item stats addition: 86

=== FOLD 4: Running Sanity Check ===
--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

[WARNING] UserID is int16, expected int. (Did a merge fail?)

[WARNING] ItemID is int16, expected int.

--- SANITY CHECK PASSED: Data is clean ---

=== FOLD 4: Optimizing Data Types ===

=== FOLD 4: DataFrame saved to /home/luigi/RecSys/xg_boost_data/dataframes/training_data/fold_4.parquet ===


In [24]:
dfs = []
for i in range(len(folds)):
    path = os.path.join(XGBOOST_DATAFRAMES, "training_data", f"fold_{i}.parquet")
    df_fold = pd.read_parquet(path, engine='fastparquet')
    dfs.append(df_fold)

In [25]:
# Merge all folds into a single DataFrame
full_df = pd.concat(dfs, ignore_index=True)

# Sort by UserID
full_df.sort_values(by=['UserID', 'ItemID'], inplace=True)

full_df

,UserID,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,...,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,13,0,0.041594,1324,0,0.239083,491,0,0.185018,...,752.299988,686.791260,1.021429,0.112598,0.103372,0.087700,1.070461,1.049027,58,357
11849335,0,13,0,0.043860,1286,0,0.253336,432,0,0.155972,...,1077.599976,913.856567,0.754376,-0.017597,0.065518,0.093295,-0.271691,1.054174,68,370
1,0,44,1,0.234533,148,0,0.552801,47,0,0.135240,...,1115.400024,2237.113037,2.100029,2.730972,0.180271,0.138595,0.880467,1.239617,58,2013
7898037,0,76,0,0.069006,855,0,0.396914,185,0,0.046504,...,304.799988,197.888214,1.310254,1.765815,0.148194,0.136293,1.099743,0.532424,68,594
7898038,0,86,0,0.314591,76,0,0.444792,133,0,0.111572,...,275.000000,475.798492,2.967940,8.367845,0.211813,0.157463,1.504626,2.449688,68,2708
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11849334,27094,6945,0,0.010339,2863,0,0.283876,281,0,0.133982,...,478.850006,614.019714,3.393603,12.987067,0.200186,0.144557,1.052063,0.610949,192,89
15797420,27094,6945,0,0.009602,2980,0,0.248066,353,0,0.131175,...,527.049988,625.409485,3.512755,13.672679,0.192101,0.149848,1.300738,1.514217,199,81
19750167,27094,6945,0,0.010335,2859,0,0.242489,376,0,0.078403,...,900.650024,1414.640869,2.243805,3.900131,0.208701,0.195603,1.095990,0.495388,211,88
7898036,27094,6948,0,0.003851,4408,0,0.089030,1206,0,0.069272,...,1662.949951,1550.113159,1.135920,-0.307864,0.105512,0.130925,2.026733,4.362759,195,33


In [26]:
# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "training_data.parquet")

full_df.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='zstd',
    index=False
)

print(f"Saved successfully to: {save_path}")

del full_df
gc.collect()

Saved successfully to: /home/luigi/RecSys/xg_boost_data/dataframes/training_data.parquet


0

## **Validation Dataframe**

In [16]:
models_folder = os.path.join(XGBOOST_MODELS, "features", "inner")

URM_train = URM_inner
URM_val = URM_outer

In [17]:
# Generate Candidates
print(f"\n=== Generating Candidates ===")
df = generate_candidates(URM_train, 
                        user_ids=np.arange(URM_train.shape[0]), 
                        models_mapping=candidate_mapping, 
                        models_cutoff=candidate_cutoff,
                        folder=models_folder)
print(f"Candidates Generated: {len(df)}")

# Add Label Column
print(f"\n=== Adding Labels ===")
# Advanced Indexing: Query the matrix directly
# "For every (User, Item) in df, give me the value in URM"
# This returns a matrix, so we convert to array and flatten
ground_truth_values = URM_val[df['UserID'].values, df['ItemID'].values]

# Convert to Boolean/Int (1 if interaction exists, 0 otherwise)
# We use np.array(..).squeeze() because sparse indexing returns a 2D structure
df['Label'] = (np.array(ground_truth_values).squeeze() > 0).astype(np.int8)

print(f"Positive Labels: {df['Label'].sum()} / {len(df)} ({100.0 * df['Label'].mean():.4f}%)")

# Add Model Features
print(f"\n=== Adding Model Features ===")
df = add_models_features(df, URM_train, models_mapping, model_folder=models_folder)
print(f"Features after model addition: {len(df.columns)}")

# Add Item-Item Similarity Features
print(f"\n=== Adding Item-Item Similarity Features ===")
df = calculate_item_item_features_fast(df, URM_train, models_folder)
print(f"Features after item-item similarity addition: {len(df.columns)}")

# Add Embedding Features
print(f"\n=== FOLD {i}: Adding Embedding Features ===")
df = add_embedding_features(df, model_folder=models_folder)
print(f"Features after embedding addition: {len(df.columns)}")

# Add Aggregate Features
print(f"\n=== Adding Aggregate Features ===")
df = add_aggregate_features_stats(df)
print(f"Features after aggregate addition: {len(df.columns)}")

# Add User Stats
print(f"\n=== Adding User and Item Stats ===")
df = add_user_stats(df, URM_train)
print(f"Features after user/item stats addition: {len(df.columns)}")

# Sanity Check
print(f"\n=== Running Sanity Check ===")
success = sanity_check(df)
if not success:
    print(f"Sanity check failed for inner fold.")

# Optimize Data Types
print(f"\n=== Optimizing Data Types ===")
df = optimize_dataframe_types(df)


=== Generating Candidates ===
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/innerSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Computing candidates with SLIMElasticNet...
Computing candidates with MultVAE...
Computing candidates with IALS...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/innerRP3beta'
RP3betaRecommender: Loading complete
Computing candidates with RP3beta...
UserKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/innerUserKNN_tversky'
UserKNNCFRecommender: Loading complete
Computing candidates with UserKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/innerItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Computing candidates with ItemKNN_tversky...
MatrixFactorization_WARP_Cython_Recommender: Loading model from file '/home/luigi/RecSys/xg_boost_data

100%|██████████| 27095/27095 [00:04<00:00, 6435.91it/s]


Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/innerRP3beta'
RP3betaRecommender: Loading complete
Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:04<00:00, 6296.34it/s]


Unloading RP3beta...
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/innerSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Extracting MAX similarity for SLIMElasticNet...


100%|██████████| 27095/27095 [00:07<00:00, 3425.21it/s]


Unloading SLIMElasticNet...
Features after item-item similarity addition: 69

=== FOLD 4: Adding Embedding Features ===
Computing distances in batches of 500...


100%|██████████| 7749/7749 [00:01<00:00, 7138.05it/s]


Assigning columns to DataFrame...
Features after embedding addition: 75

=== Adding Aggregate Features ===
Features after aggregate addition: 84

=== Adding User and Item Stats ===
Features after user/item stats addition: 86

=== Running Sanity Check ===
--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

[WARNING] UserID is int16, expected int. (Did a merge fail?)

[WARNING] ItemID is int16, expected int.

--- SANITY CHECK PASSED: Data is clean ---

=== Optimizing Data Types ===


In [18]:
df

,UserID,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,...,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,13,0,0.042611,1293,0,0.209837,546,0,0.152503,...,807.000000,640.260010,0.396104,-1.387824,0.079952,0.072270,0.379061,-1.410258,80,455
1,0,89,0,0.207530,188,0,0.542487,38,0,0.183866,...,434.700012,781.701416,2.823948,6.798428,0.181923,0.163352,1.443109,1.847162,80,2216
2,0,225,0,0.090560,642,0,0.254757,402,0,0.176012,...,785.450012,805.496155,1.316501,0.903416,0.114576,0.112987,1.198424,1.003781,80,967
3,0,227,1,0.216333,178,0,0.456547,91,0,0.345323,...,518.150024,953.186829,1.626652,0.753078,0.274519,0.167467,-0.628666,-0.755620,80,2310
4,0,275,0,0.097677,589,0,0.533726,42,0,0.302773,...,93.800003,131.963943,3.189180,11.142217,0.283331,0.143117,0.452670,-0.617746,80,1043
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3874150,27094,6624,0,0.555628,11,1,0.167659,642,0,0.213301,...,1657.050049,1946.286011,1.584614,1.367491,0.079645,0.169257,0.347632,3.573450,250,5933
3874151,27094,6724,0,0.021071,2002,0,0.322881,194,0,0.103405,...,290.000000,452.186615,3.172637,11.451586,0.280586,0.219803,1.064234,0.369629,250,225
3874152,27094,6771,0,0.383780,44,0,0.127183,889,0,0.099545,...,1709.099976,1784.538574,0.992725,-0.626230,0.094204,0.119647,-0.291556,2.981602,250,4098
3874153,27094,6810,0,0.253325,121,0,0.222065,423,0,0.080370,...,1446.400024,1776.171631,1.838771,2.804867,0.125290,0.155466,2.408205,7.939774,250,2705


In [ ]:
# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "validation_data.parquet")

df.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='zstd',
    index=False
)

print(f"Saved successfully to: {save_path}")

del df
gc.collect()

Saved successfully to: /home/luigi/RecSys/xg_boost_data/dataframes/validation_data.parquet


0

## **Prediction Dataframe**

In [20]:
models_folder = os.path.join(XGBOOST_MODELS, "features", "outer")

URM_train = URM_inner + URM_outer

In [21]:
# Generate Candidates
print(f"\n=== Generating Candidates ===")
df = generate_candidates(URM_train, 
                        user_ids=np.arange(URM_train.shape[0]), 
                        models_mapping=candidate_mapping, 
                        models_cutoff=candidate_cutoff,
                        folder=models_folder)
print(f"Candidates Generated: {len(df)}")

# Add Model Features
print(f"\n=== Adding Model Features ===")
df = add_models_features(df, URM_train, models_mapping, model_folder=models_folder)
print(f"Features after model addition: {len(df.columns)}")

# Add Item-Item Similarity Features
print(f"\n=== Adding Item-Item Similarity Features ===")
df = calculate_item_item_features_fast(df, URM_train, models_folder)
print(f"Features after item-item similarity addition: {len(df.columns)}")

# Add Embedding Features
print(f"\n=== FOLD {i}: Adding Embedding Features ===")
df = add_embedding_features(df, model_folder=models_folder)
print(f"Features after embedding addition: {len(df.columns)}")

# Add Aggregate Features
print(f"\n=== Adding Aggregate Features ===")
df = add_aggregate_features_stats(df)
print(f"Features after aggregate addition: {len(df.columns)}")

# Add User Stats
print(f"\n=== Adding User and Item Stats ===")
df = add_user_stats(df, URM_train)
print(f"Features after user/item stats addition: {len(df.columns)}")

# Sanity Check
print(f"\n=== Running Sanity Check ===")
success = sanity_check(df)
if not success:
    print(f"Sanity check failed for outer fold.")

# Optimize Data Types
print(f"\n=== Optimizing Data Types ===")
df = optimize_dataframe_types(df)


=== Generating Candidates ===
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/outerSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Computing candidates with SLIMElasticNet...
Computing candidates with MultVAE...
Computing candidates with IALS...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/outerRP3beta'
RP3betaRecommender: Loading complete
Computing candidates with RP3beta...
UserKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/outerUserKNN_tversky'
UserKNNCFRecommender: Loading complete
Computing candidates with UserKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/outerItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Computing candidates with ItemKNN_tversky...
MatrixFactorization_WARP_Cython_Recommender: Loading model from file '/home/luigi/RecSys/xg_boost_data

100%|██████████| 27095/27095 [00:04<00:00, 6499.46it/s]


Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/outerRP3beta'
RP3betaRecommender: Loading complete
Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:04<00:00, 6280.36it/s]


Unloading RP3beta...
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost_data/models/features/outerSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Extracting MAX similarity for SLIMElasticNet...


100%|██████████| 27095/27095 [00:08<00:00, 3170.22it/s]


Unloading SLIMElasticNet...
Features after item-item similarity addition: 68

=== FOLD 4: Adding Embedding Features ===
Computing distances in batches of 500...


100%|██████████| 7769/7769 [00:01<00:00, 7369.52it/s]


Assigning columns to DataFrame...
Features after embedding addition: 74

=== Adding Aggregate Features ===
Features after aggregate addition: 83

=== Adding User and Item Stats ===
Features after user/item stats addition: 85

=== Running Sanity Check ===
--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

[WARNING] UserID is int16, expected int. (Did a merge fail?)

[WARNING] ItemID is int16, expected int.

--- SANITY CHECK PASSED: Data is clean ---

=== Optimizing Data Types ===


In [22]:
df

,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,13,0.040529,1325,0,0.207680,545,0,0.129603,228,...,864.400024,707.630493,0.592600,-1.057776,0.079045,0.065471,0.427271,-1.248187,96,539
1,0,76,0.071208,820,0,0.415213,124,0,0.156571,183,...,211.000000,167.101608,2.677447,9.489000,0.178210,0.129551,1.451507,2.180224,96,947
2,0,86,0.316490,75,0,0.351859,194,0,0.099681,322,...,185.800003,202.099655,2.660424,8.649257,0.204315,0.144187,0.704453,-0.094114,96,4209
3,0,89,0.207234,188,0,0.504876,60,0,0.248379,82,...,163.600006,117.188286,2.385986,7.860408,0.194649,0.137918,0.859039,0.166705,96,2756
4,0,275,0.099481,571,0,0.514107,55,0,0.269671,72,...,97.800003,128.897995,3.014468,10.013057,0.283360,0.150004,0.427011,-0.799999,96,1323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3884153,27094,6724,0.020829,1984,0,0.342445,174,0,0.139733,232,...,264.250000,440.523712,3.500877,13.358681,0.279088,0.199187,1.070153,0.985070,314,277
3884154,27094,6771,0.390180,39,0,0.143310,811,0,0.075682,471,...,1518.900024,1675.966064,1.613655,1.880539,0.083989,0.102546,1.188648,3.077041,314,5189
3884155,27094,6782,0.010602,2757,0,0.266445,305,0,0.102625,325,...,387.000000,571.253113,4.138865,17.887188,0.211081,0.172878,1.268268,0.716687,314,141
3884156,27094,6941,0.141364,327,0,0.404935,119,0,0.273145,72,...,300.850006,621.634216,4.161278,17.983208,0.276996,0.167063,0.501362,0.499892,314,1880


In [23]:
# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "prediction_data.parquet")

df.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='zstd',
    index=False
)

print(f"Saved successfully to: {save_path}")

del df
gc.collect()

Saved successfully to: /home/luigi/RecSys/xg_boost_data/dataframes/prediction_data.parquet


0